# GPT-2 (`sdadas/polish-gpt2-medium`) + LoRA na PolEmo2

Wyniki zapisywane są na Dysku Google (`OUT`, `AUG`). Po rozłączeniu środowiska zakończone przebiegi
(z `results.json`) i gotowa augmentacja są pomijane, a przerwany trening wznawia się od checkpointu ostatniej epoki.

| Część | Wynik |
|---|---|
| klasyfikacja tekstu (LoRA, head) | `OUT/gpt2-text`, `OUT/gpt2-text-head` |
| klasyfikacja tokenów | `OUT/gpt2-token` |
| embeddingi ostatniego tokenu | `OUT/gpt2-text/plots` |
| augmentacja generatywna | `AUG/augmented_gpt2_*.json` |
| trening na augmentacji + tabela | `OUT/gpt2-text-augmented`, `OUT/gpt2_comparison.md` |

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

OUT = "/content/drive/MyDrive/nlp_lab/outputs"
AUG = "/content/drive/MyDrive/nlp_lab/augmented"

In [ ]:
%cd /content
!git clone -b gpt2 https://github.com/mtorzynska/nlp_language_models_lab.git || git -C nlp_language_models_lab pull
%cd /content/nlp_language_models_lab
# bez -U: aktualizacja moze podmienic torch z cuda na wersje bez cuda
!pip install -q "transformers>=5" "datasets>=4" peft accelerate umap-learn
# stara preinstalowana torchao psuje import peft, lora jej nie potrzebuje
!pip uninstall -y -q torchao
!python -m src.data.prepare

import torch
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "BRAK GPU")

In [ ]:
from pathlib import Path

for run in ["gpt2-text", "gpt2-text-head", "gpt2-token", "gpt2-text-augmented"]:
    path = Path(OUT) / run
    checkpoints = sorted((path / "checkpoints").glob("checkpoint-*"), key=lambda c: int(c.name.split("-")[1]))
    state = "gotowe" if (path / "results.json").exists() else f"przerwane ({checkpoints[-1].name})" if checkpoints else "brak"
    print(f"{run:22} {state}")
print(f"{'augmentacja':22} {'gotowe' if Path(AUG, 'augmented_gpt2_examples.md').exists() else 'brak'}")

## Klasyfikacja tekstu

Szybki test na małym podzbiorze i pełny trening. Batch 4 × akumulacja 4 = efektywnie 16.

In [ ]:
!python -m src.training.gpt2 --task text --max-train-samples 16 --max-eval-samples 8 --epochs 1 --max-length 64 --output-dir /content/test_run

In [ ]:
!python -m src.training.gpt2 --task text --batch-size 4 --grad-accum 4 --output-dir {OUT}/gpt2-text
!cat {OUT}/gpt2-text/test_report.txt

Baseline do porównania: zamrożony GPT-2, trenowana tylko głowica (bez LoRA). Większy learning rate, bo uczy się tylko jedna warstwa.

In [ ]:
!python -m src.training.gpt2 --task text --mode head --lr 1e-3 --batch-size 4 --grad-accum 4 --output-dir {OUT}/gpt2-text-head
!cat {OUT}/gpt2-text-head/test_report.txt

## Klasyfikacja tokenów

Każde słowo dostaje sentyment swojego zdania. Etykieta tylko na pierwszym subtokenie słowa, `<eos>` i padding mają `-100`.

In [ ]:
!python -m src.training.gpt2 --task token --batch-size 4 --grad-accum 4 --output-dir {OUT}/gpt2-token
!cat {OUT}/gpt2-token/test_report.txt

## Embeddingi ostatniego tokenu

Reprezentacja ostatniego prawdziwego tokenu (wejście głowicy klasyfikacyjnej) dla recenzji testowych:
bazowy GPT-2 (LoRA wyłączona) i po strojeniu. Wykresy UMAP i t-SNE, puste znaczniki to błędne predykcje.

In [ ]:
if not Path(f"{OUT}/gpt2-text/plots/gpt2_embeddings.npz").exists():
    !python -m src.evaluation.gpt2_embeddings --adapter {OUT}/gpt2-text/adapter

In [ ]:
from pathlib import Path
from IPython.display import HTML, display

for name in ["gpt2_base_umap", "gpt2_finetuned_umap"]:
    display(HTML(Path(f"{OUT}/gpt2-text/plots/{name}.html").read_text(encoding="utf-8")))

## Augmentacja generatywna

Początek recenzji treningowej (połowa słów, maks. 40) jest promptem, GPT-2 dopisuje resztę, a nowy tekst
dostaje etykietę recenzji źródłowej. Dwa warianty próbkowania:
`conservative` (temperature 0.7, top-k 50, top-p 0.9) i `diverse` (temperature 1.0, bez top-k, top-p 0.95).

In [ ]:
if not Path(f"{AUG}/augmented_gpt2_examples.md").exists():
    !python -m src.data.gpt2_augmentation --max-samples 2000 --output-dir {AUG}

In [ ]:
from IPython.display import Markdown

display(Markdown(Path(f"{AUG}/augmented_gpt2_examples.md").read_text(encoding="utf-8")))

## Trening na danych augmentowanych

Tak jak w części HerBERTa: train to same wygenerowane przykłady, walidacja i test bez zmian.
Pliki `augmented_mask_*_train.json` z augmentacji HerBERTa obecne w `AUG` trafiają do tego samego treningu
(jeden model na wszystkich augmentacjach).

In [ ]:
import glob

files = [f"{AUG}/augmented_gpt2_combined_train.json"] + sorted(glob.glob(f"{AUG}/augmented_mask_*_train.json"))
print(files)
AUG_FILES = " ".join(files)
!python -m src.training.gpt2 --task text --batch-size 4 --grad-accum 4 --augmented {AUG_FILES} --output-dir {OUT}/gpt2-text-augmented
!cat {OUT}/gpt2-text-augmented/test_report.txt

In [ ]:
!python -m src.evaluation.gpt2_comparison --root {OUT}
display(Markdown(Path(f"{OUT}/gpt2_comparison.md").read_text(encoding="utf-8")))